In [2]:
import numpy as np
import string

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense

# to_categorical is for one hot encoding
from tensorflow.keras.utils import to_categorical

from numpy.linalg import norm
from pathlib import Path	

In [4]:
# Sample
text = Path("sample-text.txt").read_text()

# Convert text to lowercase and tokenize
words: list[str] = text.lower().split()

print("Total words:", len(words))
print("Words:", words)

Total words: 40
Words: ['the', 'quick', 'brown', 'fox', 'jumps', 'over', 'the', 'lazy', 'dog.', 'the', 'fox', 'is', 'quick', 'and', 'the', 'dog', 'is', 'lazy.', 'machine', 'learning', 'is', 'a', 'part', 'of', 'artificial', 'intelligence.', 'deep', 'learning', 'is', 'a', 'part', 'of', 'machine', 'learning.', 'artificial', 'intelligence', 'is', 'changing', 'the', 'world.']


In [ ]:
# Remove punctuation
words = [
    word.strip(string.punctuation)
    for word in words
]

# Remove empty words
words = [word for word in words if word]

print("Total cleaned words:", len(words))
print("Cleaned words:", words)


In [ ]:
# Create vocabulary
unique_words: set[str] = set(words)

vocab: list[str] = sorted(unique_words)

vocab_size = len(vocab)

# Word to index -> used to convert words to numerical representation
word_to_index = {word: i for i, word in enumerate(vocab)}

# Index to word -> used to convert numerical representations back to words
index_to_word = {i: word for i, word in enumerate(vocab)}

print("Vocabulary Size:", vocab_size)
print("\nVocabulary:")
print(vocab)

In [ ]:
# Context window size -> used for feature representation
window_size = 2

X = [] # ! X stores input context vectors.
y = [] # ! y stores correct target words as integer indices initially.

# Generate training examples
for i in range(window_size, len(words) - window_size):
	
    context_words = (
        words[i-window_size:i] +
        words[i+1:i+window_size+1]
    )

    target_word = words[i]

    # Convert context words to one-hot vectors
    context_vector = np.zeros(vocab_size)

    for word in context_words:
        context_vector[word_to_index[word]] += 1

    # Average the context vectors
    context_vector = context_vector / len(context_words)

    # Target word index
    target_index = word_to_index[target_word]

    X.append(context_vector)
    y.append(target_index)

X = np.array(X)
y = np.array(y)

# One-hot encode target
y = to_categorical(y, num_classes=vocab_size)

print("Input shape:", X.shape)
print("Target shape:", y.shape)

In [ ]:
print("Number of training examples:", len(X))

for i in range(min(5, len(X))):
    print("\nExample", i + 1)
    print("Input vector:", X[i])
    print("Target word:", index_to_word[np.argmax(y[i])])

In [ ]:
model = Sequential([
    Dense(50, input_shape=(vocab_size,), activation='relu'),
    Dense(vocab_size, activation='softmax')
])

model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

In [ ]:
history = model.fit(
    X, y, epochs=300, batch_size=16, verbose=1
)

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 5))
plt.plot(history.history['loss'])
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("CBOW Training Loss")
plt.show()

plt.figure(figsize=(8, 5))
plt.plot(history.history['accuracy'])
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("CBOW Training Accuracy")
plt.show()

In [ ]:
def predict_word(context_words):

    context_vector = np.zeros(vocab_size)

    for word in context_words:
        if word in word_to_index:
            context_vector[word_to_index[word]] += 1

    context_vector = context_vector / len(context_words)

    prediction = model.predict(
        np.array([context_vector]),
        verbose=0
    )

    predicted_index = np.argmax(prediction)

    return index_to_word[predicted_index]


context = ["fox", "and", "the", "is"]

predicted_word = predict_word(context)

print("Context words:", context)
print("Predicted word:", predicted_word)